# Training Notebook (Spark ML, Distributed)

This notebook trains Spark ML baselines on pre-engineered features and saves outputs to both HDFS and local mounted folders.

Main outputs:
- HDFS metrics/predictions by run id
- Local metrics and preview CSV under /workspace/code/hybrid_regression_holt_winters/results/training_sparkml
- Local best model pipeline under /workspace/code/hybrid_regression_holt_winters/models/training_sparkml

Time window behavior:
- Auto-detects available min/max timestamp from feature data by default.
- Optionally set TRAIN_START_TS and TRAIN_END_TS to limit the window.

In [1]:
import json
import os
import urllib.request
import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .appName("DemandPredictionTrainingSparkML")
    .master("spark://master:7077")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def cluster_util(stage_name):
    print(f"\n===== CLUSTER UTILIZATION: {stage_name} =====")
    try:
        payload = json.load(urllib.request.urlopen("http://master:8080/json/"))
        workers = payload.get("workers", [])
        print("alive workers:", payload.get("aliveworkers"))
        print("active apps :", len(payload.get("activeapps", [])))
        for w in workers:
            print("worker", w.get("id"), "cores", f"{w.get('coresused', 0)}/{w.get('cores', 0)}", "memory", f"{w.get('memoryused', 0)}/{w.get('memory', 0)}")
    except Exception as e:
        print("Could not query Spark Master JSON:", e)

cluster_util("session_started")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d2126744-1af3-4fc6-b2ea-174856cdd50c;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 140ms :: artifacts dl 6ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------


===== CLUSTER UTILIZATION: session_started =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


In [2]:
FEATURE_PATH = "/user/data/feature_engineering/demand_prediction_features"
OUT_BASE_ROOT = "/user/data/results/demand_prediction/sparkml"
TARGET_COL = "pickup_demand"

LOCAL_BASE = "/workspace/code/hybrid_regression_holt_winters"
LOCAL_RESULTS_ROOT = f"{LOCAL_BASE}/results/training_sparkml"
LOCAL_MODELS_ROOT = f"{LOCAL_BASE}/models/training_sparkml"
LOCAL_ARTIFACTS_ROOT = f"{LOCAL_BASE}/artifacts/training_sparkml"
os.makedirs(LOCAL_RESULTS_ROOT, exist_ok=True)
os.makedirs(LOCAL_MODELS_ROOT, exist_ok=True)
os.makedirs(LOCAL_ARTIFACTS_ROOT, exist_ok=True)

# Optional manual override; keep None to auto-detect from data.
TRAIN_START_TS = None
TRAIN_END_TS = None

feature_cols = [
    "hour", "dow", "month", "is_weekend",
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12", "lag_144", "lag_1008",
    "roll_mean_6", "roll_mean_18", "roll_std_18",
]

df_all = spark.read.parquet(FEATURE_PATH)
if "pickup_bin_10m" not in df_all.columns:
    raise ValueError("Missing required column: pickup_bin_10m")

available_bounds = df_all.agg(
    F.min("pickup_bin_10m").alias("min_ts"),
    F.max("pickup_bin_10m").alias("max_ts")
).first()

if available_bounds["min_ts"] is None or available_bounds["max_ts"] is None:
    raise ValueError("Feature dataset has no usable pickup_bin_10m timestamps.")

effective_start = TRAIN_START_TS if TRAIN_START_TS is not None else available_bounds["min_ts"]
effective_end = TRAIN_END_TS if TRAIN_END_TS is not None else available_bounds["max_ts"]

window_cond = F.col("pickup_bin_10m") >= F.to_timestamp(F.lit(effective_start))
if TRAIN_END_TS is not None:
    window_cond = window_cond & (F.col("pickup_bin_10m") < F.to_timestamp(F.lit(effective_end)))
else:
    window_cond = window_cond & (F.col("pickup_bin_10m") <= F.to_timestamp(F.lit(effective_end)))

df = df_all.where(window_cond).cache()

rows = df.count()
if rows == 0:
    raise ValueError(
        f"No rows found in requested training window [{effective_start}, {effective_end}]. "
        f"Available range is [{available_bounds['min_ts']}, {available_bounds['max_ts']}]."
    )

run_id = f"{pd.Timestamp(effective_start):%Y%m%d_%H%M%S}_{pd.Timestamp(effective_end):%Y%m%d_%H%M%S}"
OUT_BASE = f"{OUT_BASE_ROOT}/{run_id}"
LOCAL_RESULTS_DIR = f"{LOCAL_RESULTS_ROOT}/{run_id}"
LOCAL_MODELS_DIR = f"{LOCAL_MODELS_ROOT}/{run_id}"
LOCAL_ARTIFACTS_DIR = f"{LOCAL_ARTIFACTS_ROOT}/{run_id}"
os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
os.makedirs(LOCAL_ARTIFACTS_DIR, exist_ok=True)

print(f"Feature rows in effective training window [{effective_start}, {effective_end}]:", rows)
df.groupBy("split").count().show()
cluster_util("after_feature_read")

# Rebuild split inside the filtered window to avoid empty train/test partitions.
bounds = df.agg(F.min("pickup_bin_10m").alias("min_ts"), F.max("pickup_bin_10m").alias("max_ts")).first()
if bounds["min_ts"] is None or bounds["max_ts"] is None:
    raise ValueError("Cannot compute local split bounds.")

min_epoch = int(bounds["min_ts"].timestamp())
max_epoch = int(bounds["max_ts"].timestamp())
cut_epoch = int(min_epoch + 0.7 * (max_epoch - min_epoch))

df = df.withColumn(
    "split_local",
    F.when(
        F.col("pickup_bin_10m") < F.to_timestamp(F.from_unixtime(F.lit(cut_epoch))),
        F.lit("train")
    ).otherwise(F.lit("test"))
).cache()

train_df = df.where(F.col("split_local") == "train")
test_df = df.where(F.col("split_local") == "test")
print("Train rows:", train_df.count())
print("Test rows :", test_df.count())

Feature rows in effective training window [2025-10-07 22:50:00, 2025-11-30 23:50:00]: 2039146
+-----+-------+
|split|  count|
+-----+-------+
|train|1427376|
| test| 611770|
+-----+-------+


===== CLUSTER UTILIZATION: after_feature_read =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


Train rows: 1427376
Test rows : 611770


In [3]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

FAST_MODE = False
TRAIN_SAMPLE_FRACTION = 0.35 if FAST_MODE else 1.0

train_fit_df = train_df
if TRAIN_SAMPLE_FRACTION < 1.0:
    train_fit_df = train_df.sample(withReplacement=False, fraction=TRAIN_SAMPLE_FRACTION, seed=42).cache()
    print(f"[FAST_MODE] Using sampled train rows: {train_fit_df.count()} / {train_df.count()}")

models = [
    (
        "linear_regression",
        LinearRegression(
            featuresCol="features",
            labelCol=TARGET_COL,
            predictionCol="prediction",
            maxIter=60 if FAST_MODE else 100,
            regParam=0.1,
            elasticNetParam=0.0,
        ),
    ),
    (
        "random_forest",
        RandomForestRegressor(
            featuresCol="features",
            labelCol=TARGET_COL,
            predictionCol="prediction",
            numTrees=80 if FAST_MODE else 180,
            maxDepth=10 if FAST_MODE else 12,
            seed=42,
        ),
    ),
    (
        "gbt",
        GBTRegressor(
            featuresCol="features",
            labelCol=TARGET_COL,
            predictionCol="prediction",
            maxIter=50 if FAST_MODE else 120,
            maxDepth=5 if FAST_MODE else 6,
            stepSize=0.05,
            seed=42,
        ),
    ),
]

mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")

metrics_rows = []
pred_union = None
fitted_models = {}

for model_name, reg in models:
    cluster_util(f"before_train_{model_name}")
    pipeline = Pipeline(stages=[assembler, reg])
    fitted = pipeline.fit(train_fit_df)
    fitted_models[model_name] = fitted

    pred = fitted.transform(test_df).withColumn(
        "prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction"))
    )

    mae_val = float(mae_eval.evaluate(pred))
    rmse_val = float(rmse_eval.evaluate(pred))
    mape_val = (
        pred.where(F.col(TARGET_COL) > 0)
        .select((F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)).alias("ape"))
        .agg((F.avg(F.col("ape")) * 100.0).alias("mape"))
        .first()["mape"]
    )

    metrics_rows.append({
        "model": model_name,
        "MAE": mae_val,
        "RMSE": rmse_val,
        "MAPE": float(mape_val) if mape_val is not None else None,
    })

    slim_pred = pred.select(
        F.lit(model_name).alias("model"),
        "PULocationID",
        "pickup_bin_10m",
        TARGET_COL,
        "prediction",
    )
    pred_union = slim_pred if pred_union is None else pred_union.unionByName(slim_pred)
    cluster_util(f"after_train_{model_name}")

metrics_pdf = pd.DataFrame(metrics_rows).sort_values("MAPE")
display(metrics_pdf)
best_model_name = metrics_pdf.iloc[0]["model"]
print("Best model by MAPE:", best_model_name)


===== CLUSTER UTILIZATION: before_train_linear_regression =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


26/04/04 05:38:25 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/04 05:38:25 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/04/04 05:38:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



===== CLUSTER UTILIZATION: after_train_linear_regression =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048

===== CLUSTER UTILIZATION: before_train_random_forest =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


26/04/04 05:38:59 WARN DAGScheduler: Broadcasting large task binary with size 1847.6 KiB
26/04/04 05:39:09 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/04/04 05:39:26 WARN DAGScheduler: Broadcasting large task binary with size 1120.3 KiB
26/04/04 05:39:27 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/04/04 05:39:50 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/04/04 05:39:51 WARN DAGScheduler: Broadcasting large task binary with size 13.9 MiB
26/04/04 05:40:26 WARN DAGScheduler: Broadcasting large task binary with size 4.3 MiB
26/04/04 05:40:30 WARN DAGScheduler: Broadcasting large task binary with size 27.2 MiB
26/04/04 05:41:00 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.2: Command exited with code 137
26/04/04 05:41:00 WARN TaskSetManager: Lost task 1.0 in stage 54.0 (TID 402) (172.18.0.2 executor 0): ExecutorLostFailure (executor 0 exited caused by one of the running tasks) Reason: Command exited w


===== CLUSTER UTILIZATION: after_train_random_forest =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048

===== CLUSTER UTILIZATION: before_train_gbt =====
alive workers: 3
active apps : 1
worker worker-20260404042505-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260404042505-172.18.0.4-7078 cores 2/2 memory 2048/2048
worker worker-20260404042557-172.18.0.2-7078 cores 2/2 memory 2048/2048


26/04/04 05:47:28 WARN DAGScheduler: Broadcasting large task binary with size 1001.6 KiB
26/04/04 05:47:29 WARN DAGScheduler: Broadcasting large task binary with size 1005.0 KiB
26/04/04 05:47:29 WARN DAGScheduler: Broadcasting large task binary with size 1005.5 KiB
26/04/04 05:47:29 WARN DAGScheduler: Broadcasting large task binary with size 1006.2 KiB
26/04/04 05:47:30 WARN DAGScheduler: Broadcasting large task binary with size 1007.2 KiB
26/04/04 05:47:30 WARN DAGScheduler: Broadcasting large task binary with size 1009.4 KiB
26/04/04 05:47:30 WARN DAGScheduler: Broadcasting large task binary with size 1014.0 KiB
26/04/04 05:47:29 WARN DAGScheduler: Broadcasting large task binary with size 1017.3 KiB
26/04/04 05:47:29 WARN DAGScheduler: Broadcasting large task binary with size 1017.8 KiB
26/04/04 05:47:29 WARN DAGScheduler: Broadcasting large task binary with size 1018.5 KiB
26/04/04 05:47:29 WARN DAGScheduler: Broadcasting large task binary with size 1019.5 KiB
26/04/04 05:47:29 WAR

Py4JError: An error occurred while calling o122.fit

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [ ]:
metrics_sdf = spark.createDataFrame(metrics_rows)
metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
pred_union.write.mode("overwrite").partitionBy("model").parquet(f"{OUT_BASE}/predictions")

best_model_path = f"{LOCAL_MODELS_DIR}/best_model_pipeline"
fitted_models[best_model_name].write().overwrite().save(best_model_path)

metrics_csv_path = f"{LOCAL_RESULTS_DIR}/metrics.csv"
metrics_json_path = f"{LOCAL_RESULTS_DIR}/metrics.json"
best_pred_csv_path = f"{LOCAL_ARTIFACTS_DIR}/best_model_predictions_preview.csv"
run_meta_path = f"{LOCAL_RESULTS_DIR}/run_metadata.json"

metrics_pdf.to_csv(metrics_csv_path, index=False)
metrics_pdf.to_json(metrics_json_path, orient="records", indent=2)

best_pred_pdf = (
    pred_union.where(F.col("model") == best_model_name)
    .orderBy("pickup_bin_10m", "PULocationID")
    .limit(5000)
    .toPandas()
)
best_pred_pdf.to_csv(best_pred_csv_path, index=False)

run_meta = {
    "run_id": run_id,
    "feature_path": FEATURE_PATH,
    "effective_start": str(effective_start),
    "effective_end": str(effective_end),
    "hdfs_output_base": OUT_BASE,
    "best_model": best_model_name,
    "best_model_local_path": best_model_path,
}
with open(run_meta_path, "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)

print("Saved HDFS outputs:")
print(f"- {OUT_BASE}/metrics")
print(f"- {OUT_BASE}/predictions")
print("Saved local outputs:")
print("-", metrics_csv_path)
print("-", metrics_json_path)
print("-", best_pred_csv_path)
print("-", best_model_path)
print("-", run_meta_path)
metrics_sdf.orderBy(F.col("MAPE").asc_nulls_last()).show(truncate=False)
cluster_util("after_output_write")

Saved outputs:
- /user/data/results/demand_prediction/2025H1_sparkml/metrics
- /user/data/results/demand_prediction/2025H1_sparkml/predictions
+------------------+-----------------+-----------------+-----------------+
|MAE               |MAPE             |RMSE             |model            |
+------------------+-----------------+-----------------+-----------------+
|1.1117127919526006|49.25495993238293|2.668239461697362|linear_regression|
|1.1929815421228893|49.9770381950957 |3.522030699013151|random_forest    |
|1.209721915178623 |51.07240693333567|3.544097277765641|gbt              |
+------------------+-----------------+-----------------+-----------------+


===== CLUSTER UTILIZATION: after_output_write =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048


In [ ]:
# Optional cleanup: run this when training is done.
spark.catalog.clearCache()
spark.stop()
print("Training Spark session stopped.")

Training Spark session stopped.
